# 第13周 · 作业：GCN 层数消融 · 预训练嵌入探针 · 复现启动（预计约7小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 研究实验：GCN 层数 {1, 2, 4, 8} × 3 个种子 + MLP 基线；结果表、误差棒图、特征行归一化讨论、实验记录 | 2 小时 |
| B | 研究实验：冻结 DistilBERT 嵌入探针 vs TF-IDF 基线（全量 + 学习曲线） | 1.5 小时 |
| C | 所选复现论文的阅读工作表 | 2 小时 |
| D | 复现计划 | 1 小时 |
| E | 代码讲解 | 0.5 小时 |

**规则**：可以使用 AI 编程助手写代码，但必须完成 E 部分的"代码讲解"（用自己的话解释关键代码，并注明哪些部分由 AI 生成），且可能被随机抽查 5 分钟口头讲解。

**评分**：A、B 按研究实验评分表（设计 30% · 执行 30% · 分析与解释 30% · 表达 10%）；C、D 看定位是否具体（节号、图表号）、原文配置是否逐项、验收标准是否可检验。

**运行时间**：A 在 CPU 上约 1 分钟（含 A7 的 12 次训练）。B 第一次运行要把 7000 条影评送进 DistilBERT（Colab GPU 约 1 分钟，CPU 3–6 分钟），结果会缓存到 `data.download_root()` 下，之后秒读。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：PyTorch Geometric（2.3 起是纯 Python 包，不需要 pyg-lib / torch-scatter 等编译扩展，也不要装它们）
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric>=2.8,<3"], check=True)
# transformers 在 Colab 已预装；本地环境缺它时兜底安装（课程按 5.x 的 API 编写）
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.22,<6"], check=True)
print("PyTorch Geometric 与 transformers 已就绪")

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", message=".*torch.jit.script.*")   # 新版 torch 对 PyG 内部写法的弃用提醒，与本课无关
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import transformers
from transformers import AutoModel, AutoTokenizer
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

from zero2researcher import data, experiments
from zero2researcher.checks import week13 as ck

gnn_device = torch.device("cpu")          # GNN 部分固定用 CPU（足够快；PyG 部分算子不支持 MPS）
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GNN device:", gnn_device, "| transformer device:", device)

## A · 研究实验：GCN 的层数

**研究问题**：在 Cora 公开划分上，GCN 的层数如何影响节点分类的测试准确率？层数加到多少时，图结构带来的好处被抵消？

实验规格：

- **自变量**：GCN 层数 $L \in \{1, 2, 4, 8\}$（第一层 1433 → 16，中间层 16 → 16，最后一层 16 → 7；每个隐层之后 ReLU + Dropout 0.5）；
- **基线**：只看节点特征的 MLP（1433 → 16 → 7，同样的 dropout）；
- **固定**：隐藏维度 16、Adam lr 0.01、weight decay 5e-4、200 epochs、不早停、原始 0/1 词袋特征、公开划分（训练 140 / 测试 1000）——与讲义 S2 完全相同。注意"原始 0/1 特征"偏离了论文 §5.2 与 PyG 教程（它们都做了行归一化）；A7 会回到这一点；
- **种子** 0、1、2；报告测试准确率的均值 ± 标准差；同时记录训练准确率（eval 模式下测），用来区分"学不会"和"学会了但泛化差"。

**A0 · 预注册**（跑任何代码之前写下，跑完不许改）。讲义已经给出了种子 0 下 MLP、2 层、8 层的结果，以及"行归一化下 8 层低于 MLP"这一条，所以这里只预注册讲义**没有**回答的问题：

1. 1 层与 4 层 GCN 各比 2 层差多少？这些差异会大于种子间的标准差吗？
2. 换成论文的行归一化特征（A7）后，MLP、2 层、4 层的测试准确率会各自变好、变差还是基本不变？4 层的种子间波动会变大还是变小？
3. 什么样的结果会推翻你的预期？

- 预期（1）：
- 预期（2）：
- 什么结果会推翻它：

In [ ]:
dataset = Planetoid(root=data.download_root(), name="Cora")     # 与讲义同一份缓存
cora = dataset[0].to(gnn_device)
print(cora)
print(f"训练 {int(cora.train_mask.sum())} | 测试 {int(cora.test_mask.sum())} 个节点")


class MLP(nn.Module):
    # 基线（讲义 S2 原样）：只看节点特征，不看引用关系
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden)
        self.lin2 = nn.Linear(hidden, out_dim)

    def forward(self, x, edge_index=None):
        x = F.dropout(self.lin1(x).relu(), p=0.5, training=self.training)
        return self.lin2(x)

**A1** · 写任意层数的 `GCN(in_dim, hidden, out_dim, num_layers)`：用 `nn.ModuleList` 装 `num_layers` 个 `GCNConv`；`forward` 里逐层计算，除最后一层外每层之后接 ReLU 与 Dropout(0.5)。`num_layers=1` 时就是一个 1433 → 7 的 `GCNConv`。

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, num_layers=2):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x, edge_index):
        # 在这里写你的代码
        raise NotImplementedError


for L in (1, 2, 8):
    m = GCN(dataset.num_features, 16, dataset.num_classes, num_layers=L)
    n_conv = sum(isinstance(mod, GCNConv) for mod in m.modules())
    print(f"{L} 层：{n_conv} 个 GCNConv，参数量 {sum(p.numel() for p in m.parameters())}")

**A2** · 写 `run(make_model, seed, epochs=200)`：固定种子 → `make_model()` 新建模型 → Adam(lr 0.01, weight decay 5e-4) 训练 `epochs` 步（整张图前向，只在训练节点上算损失）→ 在 eval 模式下返回 `(测试准确率, 训练准确率)`。**种子要在建模型之前设置**，否则初始化不受种子控制。

In [ ]:
def run(make_model, seed, epochs=200):
    # 在这里写你的代码
    raise NotImplementedError


smoke_models = []


def make_smoke_model():
    m = GCN(dataset.num_features, 16, dataset.num_classes, num_layers=2)
    smoke_models.append(m)          # 留一份引用：自查会用它在 eval 模式下重算准确率，和 run 返回的数对账
    return m


smoke_test, smoke_train = run(make_smoke_model, seed=0)
print(f"冒烟测试（2 层 GCN，种子 0）：测试 {smoke_test:.3f} | 训练 {smoke_train:.3f}")
ck.gcn_cora_accuracy(smoke_test, model=smoke_models[-1], data=cora)

**A3** · 主循环：对 `configs` 里的每个配置（MLP 与 4 种层数）跑 3 个种子，把结果存进 `results`。要求的结构（自查会检查）：

```python
results["MLP"] = {"accs": [3 个测试准确率], "mean": ..., "std": ..., "train_accs": [3 个训练准确率]}
results[L]     = {...同上...}      # L ∈ {1, 2, 4, 8}
```

`mean, std` 用 `experiments.mean_std(accs)` 算。

In [ ]:
DEPTHS, SEEDS = (1, 2, 4, 8), (0, 1, 2)
configs = {"MLP": lambda: MLP(dataset.num_features, 16, dataset.num_classes)}
for L in DEPTHS:
    # L=L 把这一轮的 L "冻"进 lambda；不写的话四个 lambda 都会用循环结束时的 L = 8（Python 闭包的经典坑）
    configs[L] = lambda L=L: GCN(dataset.num_features, 16, dataset.num_classes, num_layers=L)

results = {}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"{len(configs) * len(SEEDS)} 次训练用时 {time.time() - t0:.1f} 秒")
for key, r in results.items():
    print(f"{str(key):>4s}：测试 {experiments.format_result(r['accs'])} | 训练准确率均值 {np.mean(r['train_accs']):.3f}")
ck.gcn_depth_results(results)

**A4** · 结果表 `table`（pandas DataFrame）：行 = MLP、GCN-1、GCN-2、GCN-4、GCN-8；列 = "测试准确率（均值 ± 标准差）"与"训练准确率（均值）"。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**A5** · 误差棒图：横轴层数（1, 2, 4, 8），纵轴测试准确率（均值 ± 标准差）；MLP 基线画成水平虚线加阴影带（均值 ± 标准差）；再用另一种标记画出训练准确率。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**A6 · 分析问题**（每题两三句话，引用你表里的数字）

1. 2 层与 1 层、2 层与 4 层的差异是否大于标准差？用第 6 周的三分法（显著更好 / 显著更差 / 差异小于噪声）下结论。
2. 8 层的**训练**准确率也下降了吗？"过平滑"（节点表示变得难以区分）与"优化困难"（深网络本身难训练）各能解释哪些现象？设计一个能把二者分开的后续实验（提示：讲义里无学习的 $\hat A^k X$ 演示；Kipf & Welling 附录 B 的残差连接；更多 epochs）。
3. 本实验的固定超参数里（16 维、lr 0.01、200 epochs、每层 dropout 0.5、不早停），哪一个最可能对深层网络不公平？按第 9 周 Tuning Playbook 的说法，它应该被当作哪一类超参数？你的结论要加什么限定条件？
4. Kipf & Welling 附录 B 图 5 的结论是"2 或 3 层最好"。你的结果与之一致吗？他们的实验协议（5 折交叉验证、使用全部标签）与你的（公开划分、140 个标签）有什么不同——为什么只能比较趋势、不能比较数值？

（在这里回答）

**A7 · 讨论题：特征行归一化**（实测 + 回答）

讲义 S2 说过：Kipf & Welling §5.2 与 PyG 教程 2 都把词袋特征**行归一化**（每篇论文的特征向量除以它含的词数，`NormalizeFeatures()`），本作业沿用原始 0/1 特征。下面一格换成行归一化后的 Cora，对 MLP、GCN-2、GCN-4、GCN-8 各跑 3 个种子（复用你的 `run` 和 A3 的 `configs`；`run` 读的是全局变量 `cora`，所以只需临时换掉它），结果存进 `norm_results`（结构与 `results` 相同）。

回答（每题两三句，引用数字）：

1. 哪些配置的结论变了，哪些没变？8 层的**训练**准确率是多少——它更像"过平滑"还是"优化困难"？
2. 为什么同一种预处理对浅层几乎没影响、对 8 层影响这么大？提出一个**可检验**的假设，并说明只改哪一个变量来检验它（提示：行归一化后每个非零特征约是原来的 1/18，看看深层网络的激活与梯度尺度；也想想固定的 lr 0.01 / 200 epochs 对深层是否公平）。
3. 你在 A8 的实验记录里应该怎样限定层数消融的结论？

In [ ]:
from torch_geometric.transforms import NormalizeFeatures

cora_raw = cora
cora = Planetoid(root=data.download_root(), name="Cora", transform=NormalizeFeatures())[0].to(gnn_device)
print("行归一化后前 5 篇论文的特征和:", [round(v, 3) for v in cora.x.sum(dim=1)[:5].tolist()])
norm_results = {}
t0 = time.time()
try:
    # 在这里写你的代码
    raise NotImplementedError
finally:
    cora = cora_raw          # 换回原始特征：后面的格（以及重跑前面的格）不受影响
print(f"{len(norm_results) * len(SEEDS)} 次训练用时 {time.time() - t0:.1f} 秒")
for key, r in norm_results.items():
    print(f"{str(key):>4s}：行归一化 {experiments.format_result(r['accs'])}（训练 {np.mean(r['train_accs']):.3f}）"
          f" | 原始特征 {experiments.format_result(results[key]['accs'])}（训练 {np.mean(results[key]['train_accs']):.3f}）")

（在这里回答 A7 的三个问题）

**A8 · 实验记录**（正式提交格式，"结果"一栏写均值 ± 标准差并引用上面的表和图；"局限"一栏讨论固定超参数与特征预处理（A7）对结论的限定）

## 实验记录 #__ · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## B · 研究实验：冻结的预训练嵌入 vs TF-IDF

讲义 S3 里，只用 500 条训练数据时，冻结 DistilBERT 嵌入 + 逻辑回归明显赢了 TF-IDF。**研究问题**：用上全部 5000 条训练数据时还赢吗？如果不赢，两者在多少训练数据时交叉？

规格：模型 `distilbert-base-uncased`（冻结），`max_length=256`，两种池化（`[CLS]` 与按 mask 的 mean-pool）；分类器 `LogisticRegression(max_iter=1000)`；基线 `TfidfVectorizer(min_df=2)` + 同样的逻辑回归（与第 6 周相同，向量化器只在训练数据上 fit）；测试集固定为课程子集的 2000 条。

**B0 · 预注册**：全量时谁赢？交叉点大约在多少条训练数据？

- 预期：
- 什么结果会推翻它：

In [ ]:
imdb_train, imdb_test = data.load_imdb("train"), data.load_imdb("test")
y_train, y_test = imdb_train["label"].values, imdb_test["label"].values
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased").to(device).eval()
print(f"训练 {len(imdb_train)} 条 | 测试 {len(imdb_test)} 条 | DistilBERT {sum(p.numel() for p in bert.parameters()) / 1e6:.1f}M 参数")

**B1** · 写 `mean_pool(last_hidden_state, attention_mask)`（课堂练习 E4）与 `embed(texts, batch_size=32, max_length=256)`：按 batch 分词（`padding=True, truncation=True`）、在 `torch.no_grad()` 下取 `last_hidden_state`，返回 `([CLS] 嵌入, mean-pool 嵌入)` 两个 (N, 768) 的 numpy 数组。

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    # 在这里写你的代码
    raise NotImplementedError


def embed(texts, batch_size=32, max_length=256):
    # 在这里写你的代码
    raise NotImplementedError


ck.mean_pool(mean_pool)
c, m = embed(["A great movie!", "Dull.", "I wanted to love it, but the plot made no sense at all."])
print("冒烟测试：", c.shape, m.shape)

**B2** · 全量嵌入（下面的缓存逻辑已写好，读懂即可）：第一次运行时计算 5000 + 2000 条并存成 npz；之后文件存在就直接读。**改了 `embed` 或 `mean_pool` 之后，先删掉缓存文件再重跑**，否则读到的还是旧结果。

In [ ]:
cache_path = os.path.join(data.download_root(), "imdb_distilbert_cls_mean_len256.npz")
if os.path.exists(cache_path):
    emb = dict(np.load(cache_path))
    print("读取缓存:", cache_path)
else:
    t0 = time.time()
    train_cls, train_mean = embed(imdb_train["text"])
    test_cls, test_mean = embed(imdb_test["text"])
    emb = {"train_cls": train_cls, "train_mean": train_mean, "test_cls": test_cls, "test_mean": test_mean}
    np.savez(cache_path, **emb)
    print(f"嵌入 {len(imdb_train) + len(imdb_test)} 条用时 {time.time() - t0:.0f} 秒（{device}），已缓存到 {cache_path}")
print({k: v.shape for k, v in emb.items()})
# os.remove(cache_path)    # 需要重算时取消这一行的注释，运行一次后再注释回去

**B3** · 全量对比：在两种嵌入上各训练一个逻辑回归，再跑一遍 TF-IDF 基线，结果存进 `probe = {"cls": ..., "mean": ..., "tfidf": ...}`。

In [ ]:
probe = {}
# 在这里写你的代码
raise NotImplementedError
print(f"DistilBERT [CLS] 探针:    {probe['cls']:.4f}")
print(f"DistilBERT mean-pool 探针: {probe['mean']:.4f}")
print(f"TF-IDF + 逻辑回归（基线）:  {probe['tfidf']:.4f}")
ck.tfidf_baseline(probe["tfidf"])
ck.probe_accuracy(probe["mean"])
ck.probe_accuracy(probe["cls"])

**B4** · 学习曲线：训练数据量 $n \in \{250, 500, 1000, 2000, 5000\}$，每个 $n$ 用 3 个种子各随机抽一个训练子集（`subset(n, seed)` 已给出），在**同一个子集**上分别训练 TF-IDF、mean-pool、`[CLS]` 三种分类器，都在完整的 2000 条测试集上评估。结果存进 `curve[方法][n] = [3 个测试准确率]`。注意：TF-IDF 的向量化器要在每个子集上重新 fit。

In [ ]:
SIZES, SEEDS = (250, 500, 1000, 2000, 5000), (0, 1, 2)


def subset(n, seed):
    # 从 5000 条训练数据里随机抽 n 条的下标（保持原有顺序）；n = 5000 时就是全量
    return np.sort(np.random.default_rng(seed).permutation(len(imdb_train))[:n])


curve = {"TF-IDF": {}, "mean-pool": {}, "[CLS]": {}}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"{len(SIZES) * len(SEEDS) * len(curve)} 个分类器用时 {time.time() - t0:.0f} 秒")
for method, by_n in curve.items():
    print(f"{method:9s} " + " | ".join(f"n={n}: {experiments.format_result(a, 3)}" for n, a in by_n.items()))

**B5** · 画学习曲线：横轴训练数据量（对数刻度），纵轴测试准确率（均值 ± 标准差，误差棒），三种方法各一条线。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**B6** · 跨周 IMDB 对比表（列名沿用第 10 周：`model` / `test_acc` / `week`）。如果你手上有第 10 周起保存的 `outputs/imdb_results.csv`（在 Colab 上把它上传到本 notebook 旁边的 `outputs/` 文件夹），下面会直接读它；没有的话，把你自己第 10 周 TextCNN、第 11 周 Transformer 的测试准确率填进 `my_previous`（没有就留 `None`）：

In [ ]:
board_path = os.path.join("outputs", "imdb_results.csv")        # 第 10 周 notebook 保存的跨周对比表
if os.path.exists(board_path):
    board = pd.read_csv(board_path)
    print("读到之前各周的对比表:", board_path)
else:
    my_previous = [("TextCNN", None, 10), ("Transformer (from scratch)", None, 11)]   # 例如 ("TextCNN", 0.78, 10)
    rows = [{"model": "TF-IDF + LR", "test_acc": probe["tfidf"], "week": 6}]
    rows += [{"model": name, "test_acc": acc, "week": week} for name, acc, week in my_previous if acc is not None]
    board = pd.DataFrame(rows)
week13_rows = pd.DataFrame([
    {"model": "DistilBERT [CLS] probe (frozen)", "test_acc": round(probe["cls"], 4), "week": 13},
    {"model": "DistilBERT mean-pool probe (frozen)", "test_acc": round(probe["mean"], 4), "week": 13},
])
board = (pd.concat([board, week13_rows], ignore_index=True)
         .drop_duplicates("model", keep="last")                 # 重复运行时不会出现两行同名
         .sort_values("test_acc", ascending=False).reset_index(drop=True))
board

**B7 · 分析问题**（每题两三句话，引用数字）

1. 全量（$n = 5000$）时，两种探针与 TF-IDF 的差距各是多少？$n = 5000$ 时 3 个"种子"给出完全相同的结果、标准差为 0——这里的"种子"控制的是哪一种随机性？全量时如果要给差距一个误差范围，你会怎么做（提示：第 9 周讲过的 bootstrap，对**测试集**重抽样）？
2. 交叉点大约在多少条训练数据？为什么数据少时预训练嵌入占优、数据多时优势消失？
3. 至少给出两个"冻结嵌入在全量时赢不了 TF-IDF"的解释。提示：`max_length=256` 截断了多少评论（TF-IDF 看的是全文）？嵌入是为什么任务训练的？
4. 我们对两边都用了逻辑回归的默认正则化强度 `C=1`。按第 9 周的说法 `C` 是什么类型的超参数？公平的比较应该怎么处理它？（选做：从训练集里切 1000 条作验证集，为两边各自选 `C ∈ {0.1, 1, 10}`，看结论变不变）
5. 把本周结果和第 10、11 周放进同一张表（B6），你得出什么结论？它为第 14 周的"微调"提出了什么问题？

（在这里回答）

## C · 论文阅读工作表

从讲义 S4 的三篇候选论文中选一篇（第 15 周你要汇报对它的复现），按三遍阅读法读完，填写下面的工作表（与 `templates/paper_reading_worksheet.md` 相同）。

- ① Adam：Kingma & Ba, ICLR 2015 — [arXiv:1412.6980](https://arxiv.org/abs/1412.6980)，复现图 2(a)
- ② Dropout：Srivastava et al., JMLR 2014 — [jmlr.org/papers/v15/srivastava14a.html](https://jmlr.org/papers/v15/srivastava14a.html)，复现表 2 或图 4
- ③ ResNet：He et al., CVPR 2016 — [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)，复现图 6 左的简化版（plain-20 vs plain-56）

要求：定位表每一行都写**节号或图表号**；"一句话"用自己的话写，不抄原文；"评价指标与训练配置"一行写到复现时能直接照着设的程度。

## 论文阅读工作表 · 日期：____

> 依据 S. Keshav, "How to Read a Paper", *ACM SIGCOMM Computer Communication Review* 37(3), 2007 的三遍阅读法。
> 每一遍读完再往下填；第一遍之后可以决定"不再读下去"——这也是合法的结论。

- 论文（作者，标题，会议/期刊，年份，arXiv 号或链接）：
- 我为什么读它（要解决我的什么问题）：

### 第一遍（5–10 分钟）：鸟瞰

只读标题、摘要、引言，各节与小节的标题，结论；扫一眼参考文献里有哪些你认识的。读完回答 Keshav 的 **5C**：

- Category（类别）：这是哪一类论文？（新方法 / 分析已有方法 / 新数据集或基准 / 系统 / 综述）
- Context（背景）：它接着哪些已有工作做？依据了什么理论或已有方法？
- Correctness（正确性）：它的前提假设看起来成立吗？
- Contributions（贡献）：作者声称的主要贡献（最多 3 条，用自己的话写）：
- Clarity（清晰度）：写得清楚吗？哪里读不懂？
- 第一遍结论：值不值得读第二遍？为什么？

### 第二遍（约 1 小时）：抓住内容，跳过证明

- 逐一过关键图表：坐标轴是什么、单位是什么、有没有误差棒（几个种子/几次运行）？图表真的支持正文的说法吗？
- 正文里**没读懂的术语**（列出来，回头查）：
- 需要追读的参考文献（最多 3 篇，写明为什么）：
- 用三五句话向同学复述这篇论文（不看原文）：

### 第三遍（复现者视角，数小时）：在脑中重做这项工作

- 如果由我来做，我会怎样设计实验？与作者的做法差在哪里？
- 隐含的假设、没写出来的细节（复现时最可能卡住的地方）：
- 我能想到的弱点或替代解释：
- 可以做的后续工作：

### 定位表（第二遍完成）

每一行写清**在哪里**（节号 / 页码 / 图表号）和**一句话**（用自己的话，不抄原文）。

| 要素 | 位置 | 一句话 |
| --- | --- | --- |
| 问题（要解决什么、为什么难） |  |  |
| 主张（作者声称做到了什么） |  |  |
| 方法（核心想法，一句话能说清的那部分） |  |  |
| 基线（和谁比；基线是否调得公平） |  |  |
| 消融（拿掉哪个部件、结果怎样） |  |  |
| 数据与划分（数据集、训练/验证/测试怎么分、有无泄漏风险） |  |  |
| 关键图/表（支撑主张的那一张） |  |  |
| 评价指标与训练配置（复现时要用） |  |  |

### 我准备复现的那张图/表

- 编号与内容：
- 它支撑上表中的哪条主张：
- 原文报告的数值或曲线形状：

## D · 复现计划

按模板写出你的复现计划（与 `templates/replication_plan.md` 相同）。第 14 周 S4 的进度检查会对照这份计划：你应该能带着第一条曲线来。

## 复现计划 · 日期：____

> 先读完论文并填好 `paper_reading_worksheet.md` 再写本计划。原则：**先写下预期，再动手跑**——
> 跑完才定的"验收标准"不叫验收标准。时间线：第 13 周选题 → 第 14 周进度检查 → 第 15 周汇报。

- 论文（作者，标题，会议/期刊，年份）：
- 复现目标（哪一张图或哪一个表，写编号；只选一个）：
- 它支撑原文的哪条主张：

### 原文设置（逐项抄出，并注明出自原文第几节；原文没写的写"未给出"）

- 模型 / 架构：
- 数据集与预处理：
- 训练 / 验证 / 测试划分：
- 优化器、学习率及其日程：
- batch size、训练轮数（epochs）或步数：
- 正则化（dropout、权重衰减、数据增强……）：
- 运行次数 / 随机种子：
- 评价指标（训练误差还是测试误差？准确率还是损失？）：

### 我的设置

- 与原文的每一处差异（例如缩短训练、缩小网络、换框架），以及理由：
- 数据（来源、下载方式、划分）：
- 随机种子与统计（几个种子；报告均值 ± 标准差）：
- 基线 / 对照组：

### 预期与验收

- 原文报告的数值或曲线形状：
- 在我的缩短设置下预期的数值范围：
- **复现的是趋势还是数值**（二选一）：
- 验收标准（什么结果算"复现成功"，什么结果算"没复现出来"——两者都要写）：

### 预算与风险

- 时间预算（Colab T4）：单次运行约 ___ 分钟 × ___ 次 = ___ 分钟；另留调试时间 ___
- 最大风险（最可能让你复现失败的一件事）：
- 应对办法：
- 如果最终没复现出来，我会怎样判断是"我的实现有问题"还是"原文结论在我的设置下不成立"：

## E · 代码讲解（每份作业必交）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 A 部分的 `GCN.forward` 与 `run`——尤其是：为什么损失只在训练节点上算、前向却要用整张图？为什么 `configs` 里要写 `lambda L=L`？
2. 解释 B 部分的缓存逻辑：什么情况下缓存会让你得到**错误**的结果？
3. 注明哪些代码由 AI 助手生成、你做了哪些检查或修改（例如：种子是否在建模型之前设置、评估时是否 `model.eval()`、mean-pool 是否用了 mask、TF-IDF 是否只在训练子集上 fit）。

讲不清楚的代码视同不是你写的——口头抽查就从这一段问起。

（在这里写）